<a href="https://colab.research.google.com/github/RushiKP14/Tensorflow/blob/main/fcc_book_recommendation_knn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
# import libraries (you may add additional imports but you may not have to)
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

In [11]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/books/book-crossings.zip

!unzip book-crossings.zip

books_filename = 'BX-Books.csv'
ratings_filename = 'BX-Book-Ratings.csv'

--2025-03-06 14:39:55--  https://cdn.freecodecamp.org/project-data/books/book-crossings.zip
Resolving cdn.freecodecamp.org (cdn.freecodecamp.org)... 104.26.2.33, 104.26.3.33, 172.67.70.149, ...
Connecting to cdn.freecodecamp.org (cdn.freecodecamp.org)|104.26.2.33|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26085508 (25M) [application/zip]
Saving to: ‘book-crossings.zip’

book-crossings.zip  100%[===================>]  24.88M  60.2MB/s    in 0.4s    

2025-03-06 14:39:56 (60.2 MB/s) - ‘book-crossings.zip’ saved [26085508/26085508]

Archive:  book-crossings.zip
  inflating: BX-Book-Ratings.csv     
  inflating: BX-Books.csv            
  inflating: BX-Users.csv            


In [32]:
# import csv data into dataframes
df_books = pd.read_csv(
    books_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['isbn', 'title', 'author'],
    usecols=['isbn', 'title', 'author'],
    dtype={'isbn': 'str', 'title': 'str', 'author': 'str'})

df_ratings = pd.read_csv(
    ratings_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    dtype={'user': 'int32', 'isbn': 'str', 'rating': 'float32'})

In [71]:
# add your code here - consider creating a new cell for each section of code
df_books.head()
df_ratings.head()
#df_ratings.shape
#df_ratings.describe()

,user,isbn,rating
0,276725,034545104X,0.0
1,276726,0155061224,5.0
2,276727,0446520802,0.0
3,276729,052165615X,3.0
4,276729,0521795028,6.0


In [30]:
import pandas as pd

# create a sample DataFrame
df = pd.DataFrame({
    'order_id': [1, 2, 3, 4],
    'customer_id': [101, 102, 103, 104],
    'product_name': ['Coca Cola', 'Pepsi', 'Fanta', 'Sprite'],
    'quantity': [2, 1, 3, 2]
})
print(df)
mask = df['product_name'] == 'Coca Cola'

# select all rows except the ones that contain 'Coca Cola'
df = df[~mask]

# print the resulting DataFrame
print(df)

   order_id  customer_id product_name  quantity
0         1          101    Coca Cola         2
1         2          102        Pepsi         1
2         3          103        Fanta         3
3         4          104       Sprite         2
   order_id  customer_id product_name  quantity
1         2          102        Pepsi         1
2         3          103        Fanta         3
3         4          104       Sprite         2
  product_name
1        Pepsi
2        Fanta
3       Sprite


In [60]:
list_user = df_ratings.user.value_counts()
print(list_user)
keep_user=[]
for i in list_user.index:
  if list_user.get(i)>=200:
    keep_user.append(i)
#print(keep_user)
flag=1
for user in keep_user:
  mask = df_ratings['user'] == user
  if flag==1:
    df_ratings_new = df_ratings.loc[mask]
    flag=0
  else:
    df_ratings_new = pd.concat([df_ratings_new, df_ratings.loc[mask]], ignore_index=True)
print(df_ratings_new)
sum(list_user.get(keep_user))

user
11676     13602
198711     7550
153662     6109
98391      5891
35859      5850
          ...  
116180        1
116166        1
116154        1
116137        1
276723        1
Name: count, Length: 105283, dtype: int64


,user,isbn,rating
0,11676,9022906116,7.0
1,11676,"\0432534220\""""",6.0
2,11676,"\2842053052\""""",7.0
3,11676,0 7336 1053 6,0.0
4,11676,0=965044153,7.0
...,...,...,...
527551,26883,3401015834,7.0
527552,26883,3423000015,0.0
527553,26883,3442720117,7.0
527554,26883,3822505986,0.0


In [73]:
list_isbn = df_ratings_new.isbn.value_counts()
print(list_isbn)
keep_isbn=[]
for i in list_isbn.index:
  if list_isbn.get(i)>=100:
    keep_isbn.append(i)
print(keep_isbn)
flag=1
for isbn in keep_isbn:
  mask = df_ratings_new['isbn'] == isbn
  if flag==1:
    df_ratings_final = df_ratings_new.loc[mask]
    flag=0
  else:
    df_ratings_final = pd.concat([df_ratings_final, df_ratings_new.loc[mask]], ignore_index=True)
print(df_ratings_final)
sum(list_isbn.get(keep_isbn))

isbn
0971880107    365
0316666343    272
0060928336    221
0440214041    218
0385504209    217
             ... 
0340788658      1
0340770643      1
0340718129      1
0340710640      1
3929017245      1
Name: count, Length: 207699, dtype: int64
['0971880107', '0316666343', '0060928336', '0440214041', '0385504209', '044021145X', '0440211727', '067976402X', '0446672211', '059035342X', '0440222656', '0679781587', '0345337662', '0804106304', '0316601950', '0312195516', '0671027360', '0446605239', '0743418174', '0345370775', '044023722X', '0142001740', '0440226430', '0375727345', '0446606812', '006101351X', '0060976845', '0440213525', '0375706771', '0553279912', '0446310786', '0440221471', '0440220602', '044022165X', '0440206154', '0440225701', '0345361792', '0312278586', '1400034779', '0452282152', '044651652X', '0553268880', '0060930535', '0156027321', '0440224675', '0671003755', '0446364193', '0440236673', '0553280341', '068484477X', '0671021001', '0446610038', '080410526X', '0671001795'

,user,isbn,rating
0,11676,0971880107,6.0
1,198711,0971880107,0.0
2,153662,0971880107,0.0
3,35859,0971880107,0.0
4,76352,0971880107,0.0
...,...,...,...
13788,206074,0515131229,0.0
13789,132083,0515131229,0.0
13790,20201,0515131229,0.0
13791,242646,0515131229,0.0


In [ ]:
# function to return recommended books - this will be tested
def get_recommends(book = ""):


  return recommended_books

In [ ]:
books = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
print(books)

def test_book_recommendation():
  test_pass = True
  recommends = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
  if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
    test_pass = False
  recommended_books = ["I'll Be Seeing You", 'The Weight of Water', 'The Surgeon', 'I Know This Much Is True']
  recommended_books_dist = [0.8, 0.77, 0.77, 0.77]
  for i in range(2):
    if recommends[1][i][0] not in recommended_books:
      test_pass = False
    if abs(recommends[1][i][1] - recommended_books_dist[i]) >= 0.05:
      test_pass = False
  if test_pass:
    print("You passed the challenge! 🎉🎉🎉🎉🎉")
  else:
    print("You haven't passed yet. Keep trying!")

test_book_recommendation()